In [1]:
import json
from abc import ABC, abstractmethod
from datetime import datetime
import os
# Geliştiriciler: "Abdülhalik , Emirhan ve Miraç"
class Arac(ABC):
    """
    Soyutlama (Abstraction): Temel araç sınıfımız.
    Bu sınıftan doğrudan nesne üretilmez, diğer araç tiplerine şablon olur.
    """
    def __init__(self, plaka, giris_zamani=None):
        # Kapsülleme (Encapsulation): Plaka dışarıdan doğrudan değiştirilemez, private yapıldı.
        self.__plaka = plaka

        if giris_zamani:
            self.giris_zamani = datetime.strptime(giris_zamani, "%Y-%m-%d %H:%M:%S")
        else:
            self.giris_zamani = datetime.now()
    def get_plaka(self):
        """ Kapsüllenmiş plaka verisine kontrollü erişim sağlar. """
        return self.__plaka
    @abstractmethod
    def ucret_hesapla(self, kalinan_saat):
        """
        Çok Biçimlilik (Polymorphism) için alt sınıflarda ezilecek (override) soyut metot.
        Her aracın fiyatlandırması farklı olacaktır.
        """
        pass

    def to_dict(self):
        """ Verileri dosyaya kaydederken sözlüğe dönüştürmek için kolaylık. """
        return {
            "tip": self.__class__.__name__,
            "plaka": self.__plaka,
            "giris_zamani": self.giris_zamani.strftime("%Y-%m-%d %H:%M:%S")
        }
class Otomobil(Arac):
    """
    Kalıtım (Inheritance): Otomobil sınıfı Arac sınıfından miras alır.
    """
    def __init__(self, plaka, giris_zamani=None):
        super().__init__(plaka, giris_zamani)
        self.saatlik_ucret = 20.0
    def ucret_hesapla(self, kalinan_saat):
        """
        Çok Biçimlilik (Polymorphism): Otomobil için ücret hesaplama.
        1 saat veya altı için asgari ücret ödenir.
        """
        if kalinan_saat <= 1:
            return self.saatlik_ucret
        return kalinan_saat * self.saatlik_ucret
class Motosiklet(Arac):
    """
    Kalıtım (Inheritance): Motosiklet sınıfı Arac sınıfından miras alır.
    """
    def __init__(self, plaka, giris_zamani=None):
        super().__init__(plaka, giris_zamani)
        self.saatlik_ucret = 10.0
    def ucret_hesapla(self, kalinan_saat):
        """
        Çok Biçimlilik (Polymorphism): Motosiklet için ücret hesaplama.
        1 saat veya altı için asgari ücret ödenir.
        """
        if kalinan_saat <= 1:
            return self.saatlik_ucret
        return kalinan_saat * self.saatlik_ucret
class TicariArac(Arac):
    """
    Kalıtım (Inheritance): TicariArac sınıfı Arac sınıfından miras alır.
    """
    def __init__(self, plaka, giris_zamani=None):
        super().__init__(plaka, giris_zamani)
        self.saatlik_ucret = 35.0
    def ucret_hesapla(self, kalinan_saat):
        """
        Çok Biçimlilik (Polymorphism): Ticari araç için ücret hesaplama.
        1 saat veya altı için asgari ücret ödenir.
        """
        if kalinan_saat <= 1:
            return self.saatlik_ucret
        return kalinan_saat * self.saatlik_ucret
class Otopark:
    """
    Otopark yönetimini sağlayan ve araçları kontrol eden sınıf.
    """
    def __init__(self):
        # Veri Yönetimi: Araçları plaka anahtarına göre bir sözlükte (dictionary) tutuyoruz.
        self.araclar = {}
        # Kapsülleme (Encapsulation): Günlük gelir bilgisi dışarıdan müdahaleye kapalı.
        self.__gunluk_gelir = 0.0
        self.dosya_adi = "otopark_verisi.json"
    def get_gunluk_gelir(self):
        return self.__gunluk_gelir
    def arac_girisi(self, tip, plaka):
        # Kapsülleme ve Hata Yönetimi: Var olan bir aracı tekrar eklemeyi engelliyoruz.
        if plaka in self.araclar:
            raise ValueError(f"{plaka} plakalı araç zaten otoparkta bulunuyor!")
        if tip == "1":
            yeni_arac = Otomobil(plaka)
        elif tip == "2":
            yeni_arac = Motosiklet(plaka)
        elif tip == "3":
            yeni_arac = TicariArac(plaka)
        else:
            raise ValueError("Geçersiz araç tipi seçildi!")
        self.araclar[plaka] = yeni_arac
        print(f"==> {plaka} plakalı {yeni_arac.__class__.__name__} girişi yapıldı. Saat: {yeni_arac.giris_zamani.strftime('%H:%M:%S')}")
    def araclari_listele(self):
        if not self.araclar:
            print("Otopark şu an boş.")
            return
        print("\n--- Otoparktaki Araçlar ---")
        for plaka, arac in self.araclar.items():
            print(f"Tip: {arac.__class__.__name__:12} | Plaka: {plaka:10} | Giriş: {arac.giris_zamani.strftime('%H:%M:%S')}")
        print("---------------------------")
    def arac_sorgula(self, plaka):
        if plaka in self.araclar:
            arac = self.araclar[plaka]
            print(f"Araç Bulundu! Tip: {arac.__class__.__name__} | Giriş: {arac.giris_zamani.strftime('%H:%M:%S')}")
        else:
            print(f"Uyarı: {plaka} plakalı araç otoparkta bulunamadı.")
    def arac_cikisi(self, plaka):
        # Hata Yönetimi: Olmayan plakayı çıkarmaya çalışmak bir ValueError fırlatır.
        if plaka not in self.araclar:
            raise ValueError(f"Sistemde {plaka} plakalı araç bulunamadığından çıkış işlemi yapılamaz!")
        arac = self.araclar.pop(plaka)
        cikis_zamani = datetime.now()

        # Testlerde çıkışların çok hızlı olacağı göz önüne alınarak, farkın sıfır olmaması sağlanır
        fark = cikis_zamani - arac.giris_zamani
        kalinan_saat = fark.total_seconds() / 3600.0

        # Çok Biçimlilik (Polymorphism): arac nesnesi Otomobil ise farklı, TicariArac ise farklı çalışır.
        ucret = arac.ucret_hesapla(kalinan_saat)

        self.__gunluk_gelir += ucret

        print(f"\n--- Çıkış İşlemi Başarılı ---")
        print(f"Plaka             : {plaka}")
        print(f"K kalınan Süre    : {kalinan_saat:.4f} Saat")
        print(f"Ödenecek Ücret    : {ucret:.2f} TL")
        print("-----------------------------")
    def verileri_kaydet(self):
        # Hata Yönetimi: Dosya yazma işlemlerindeki olası yetki veya disk hatalarına karşı try-except
        try:
            veri = {
                "araclar": [arac.to_dict() for arac in self.araclar.values()],
                "gunluk_gelir": self.__gunluk_gelir
            }
            with open(self.dosya_adi, "w", encoding="utf-8") as f:
                json.dump(veri, f, ensure_ascii=False, indent=4)
            print("=> Veriler JSON dosyasına başarıyla kaydedildi.")
        except Exception as e:
            print(f"Kayıt sırasında teknik bir hata oluştu: {e}")
    def verileri_yukle(self):
        # Veri Yönetimi: Başlangıçta .json üzerinden araçlar ve bakiye yüklenir.
        if not os.path.exists(self.dosya_adi):
            print("Mevcut veri dosyası bulunamadı, sistem temiz bir başlangıç yapıyor.")
            return
        try:
            with open(self.dosya_adi, "r", encoding="utf-8") as f:
                veri = json.load(f)

            self.araclar.clear()
            self.__gunluk_gelir = veri.get("gunluk_gelir", 0.0)

            for arac_veri in veri.get("araclar", []):
                tip = arac_veri["tip"]
                plaka = arac_veri["plaka"]
                giris = arac_veri["giris_zamani"]

                # Polimorfik yeniden yapılandırma (Veriye göre uygun alt sınıf üretilir)
                if tip == "Otomobil":
                    self.araclar[plaka] = Otomobil(plaka, giris)
                elif tip == "Motosiklet":
                    self.araclar[plaka] = Motosiklet(plaka, giris)
                elif tip == "TicariArac":
                    self.araclar[plaka] = TicariArac(plaka, giris)

            print("=> Veriler başarıyla dosyadan yüklendi.")
        except Exception as e:
            print(f"Yükleme sırasında hata oluştu, veri bozulmuş olabilir: {e}")
def main():
    otopark = Otopark()
    otopark.verileri_yukle() # Program ilk açıldığında verileri otomatik yükle

    while True:
        print("\n=== AKILLI OTOPARK YÖNETİM SİSTEMİ ===")
        print("1- Araç Girişi Yap")
        print("2- İçerideki Araçları Listele")
        print("3- Plaka ile Araç Sorgula")
        print("4- Araç Çıkışı ve Ücret Hesapla")
        print("5- Günlük Gelir Raporunu Gör")
        print("6- Verileri Dosyaya Kaydet")
        print("7- Verileri Dosyadan Yükle")
        print("0- Çıkış")
        print("=======================================")

        try:
            # Hata Yönetimi: Menü seçiminde kullanıcının geçersiz bir giriş yapmasına karşı try-except
            secim = input("Lütfen işleminizi seçiniz: ").strip()

            if secim == "1":
                print("\nAraç Tipleri:")
                print("1- Otomobil")
                print("2- Motosiklet")
                print("3- Ticari Araç")
                tip = input("Araç tipini seçiniz (1/2/3): ").strip()
                if tip not in ["1", "2", "3"]:
                    raise ValueError("Araç tipi olarak sadece 1, 2 veya 3 girilebilir!")

                plaka = input("Aracın Plakasını Giriniz (Örn: 34ABC123): ").upper().replace(" ", "")
                if not plaka:
                    raise ValueError("Plaka boş bırakılamaz!")

                otopark.arac_girisi(tip, plaka)

            elif secim == "2":
                otopark.araclari_listele()

            elif secim == "3":
                plaka = input("Sorgulanacak Plakayı Giriniz: ").upper().replace(" ", "")
                if not plaka:
                    raise ValueError("Plaka bilgisi eksik!")
                otopark.arac_sorgula(plaka)

            elif secim == "4":
                plaka = input("Çıkış Yapacak Aracın Plakası: ").upper().replace(" ", "")
                if not plaka:
                    raise ValueError("Plaka bilgisi eksik!")
                # Çıkarılmaya çalışılan plaka yoksa, sınıf içindeki exception fırlatılır ve alttaki ValueError bloğuna düşer
                otopark.arac_cikisi(plaka)

            elif secim == "5":
                print(f"\n-> Şu ana kadar elde edilen toplam günlük gelir: {otopark.get_gunluk_gelir():.2f} TL")

            elif secim == "6":
                otopark.verileri_kaydet()

            elif secim == "7":
                otopark.verileri_yukle()

            elif secim == "0":
                print("Çıkış yapılıyor... Veriler otomatik kaydediliyor.")
                otopark.verileri_kaydet()
                print("İyi günler dileriz!")
                break

            else:
                print("Hata: Geçersiz bir menü seçimi yaptınız. Lütfen 0-7 arası bir rakam giriniz.")

        except ValueError as ve:
            # Hata Yönetimi: Mantıksal hataları ve kullanıcı veri hatalarını (olmayan plaka silme, hatalı harf girme) yakalama
            print(f"HATA: {ve}")
        except KeyboardInterrupt:
            print("\nProgram zorla kapatıldı. Kapatılmadan önce veriler kaydediliyor...")
            otopark.verileri_kaydet()
            break
        except Exception as e:
            # Hata Yönetimi: Beklenmeyen diğer tüm teknik hataları uygulamanın çökmemesi için yakalama
            print(f"BEKLENMEYEN HATA OLUŞTU: {e}")
if __name__ == "__main__":
    main()


Mevcut veri dosyası bulunamadı, sistem temiz bir başlangıç yapıyor.

=== AKILLI OTOPARK YÖNETİM SİSTEMİ ===
1- Araç Girişi Yap
2- İçerideki Araçları Listele
3- Plaka ile Araç Sorgula
4- Araç Çıkışı ve Ücret Hesapla
5- Günlük Gelir Raporunu Gör
6- Verileri Dosyaya Kaydet
7- Verileri Dosyadan Yükle
0- Çıkış
Lütfen işleminizi seçiniz: 5

-> Şu ana kadar elde edilen toplam günlük gelir: 0.00 TL

=== AKILLI OTOPARK YÖNETİM SİSTEMİ ===
1- Araç Girişi Yap
2- İçerideki Araçları Listele
3- Plaka ile Araç Sorgula
4- Araç Çıkışı ve Ücret Hesapla
5- Günlük Gelir Raporunu Gör
6- Verileri Dosyaya Kaydet
7- Verileri Dosyadan Yükle
0- Çıkış

Program zorla kapatıldı. Kapatılmadan önce veriler kaydediliyor...
=> Veriler JSON dosyasına başarıyla kaydedildi.
